In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt
from skimpy import skim
from IPython.display import Markdown as md
import folium
from nbconvert.exporters import HTMLExporter
from nbconvert import PDFExporter
import nbformat
from traitlets.config import Config
from nbconvert.preprocessors import ExecutePreprocessor

# Run data downloader - download newest data file, merge with other data files and clean these files.

In [ ]:
exec(open("./csvdata_downloader.py").read())

### Data frame creator

In [ ]:
df_smog = pd.read_csv('../data/smog_merged.csv', encoding='utf-8-sig')

### Rzutowanie kolumn tekstowych na numeryczne (błędy zamieniane na NaN)

In [ ]:
cols_to_numeric = ['HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG']
for col in cols_to_numeric:
    df_smog[col] = df_smog[col].astype(str).str.replace(',', '.')
    df_smog[col] = pd.to_numeric(df_smog[col], errors='coerce')

### Usunięcie wierszy z brakami danych (NaN) w kluczowych kolumnach

In [ ]:
df_smog = df_smog.dropna(subset=['PM10_AVG', 'PM25_AVG', 'LATITUDE', 'LONGITUDE', 'HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG'])

### Set variables

In [ ]:
area_dict = {
    range(0, 10): 'Mazowieckie',
    range(10, 15): 'Warmińsko-mazurskie',
    range(15, 20): 'Podlaskie',
    range(20, 25): 'Lubelskie',
    range(25, 30): 'Świętokrzyskie',
    range(30, 35): 'Małopolskie',
    range(35, 40): 'Podkarpackie',
    range(40, 45): 'Śląskie',
    range(45, 50): 'Opolskie',
    range(50, 60): 'Dolnośląskie',
    range(60, 65): 'Wielkopolskie',
    range(65, 70): 'Lubuskie',
    range(70, 79): 'Zachodniopomorskie',
    range(80, 85): 'Pomorskie',
    range(85, 90): 'Kujawsko-pomorskie',
    range(90, 100): 'Łódzkie'
}
smog_columns = ['LONGITUDE', 'LATITUDE', 'HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG', 'PM10_AVG', 'PM25_AVG']

### Zmiana typu danych w kolumnie 'Date' na datetime

In [ ]:
df_smog['TIMESTAMP_DATETIME'] = pd.to_datetime(df_smog['TIMESTAMP'], errors='coerce')

### Wyciągnięcie daty (Format: YYYY-MM-DD)

In [ ]:
df_smog['TYLKO_DATA'] = df_smog['TIMESTAMP_DATETIME'].dt.date

### Wyciągnięcie samego czasu (Format: GG:MM:SS)

In [ ]:
df_smog['TYLKO_CZAS'] = df_smog['TIMESTAMP_DATETIME'].dt.time

### Usunięcie kolumny pomocniczej

In [ ]:
df_smog = df_smog.drop(columns=['TIMESTAMP_DATETIME'])

# Insert area column based on post code

In [ ]:
def set_area(area_value):
    for x, y in area_dict.items():
        if area_value in x:
            area_value = y
    return area_value
area = []
for x, y in df_smog['POST_CODE'].astype(str).str[:3].astype(int).items():
    area.append(set_area(y))
df_smog.insert(2, 'AREA', area)

### Nazwy nagłówków

In [ ]:
print(df_smog.columns)

### Strip '-' from post code

In [ ]:
for x in df_smog['POST_CODE']:
    df_smog['POST_CODE'] = df_smog['POST_CODE'].str.replace("-", "")

# Sort values by area

In [ ]:
df_smog = df_smog.sort_values(['POST_CODE', 'CITY', 'STREET'])

### Opis danych

In [ ]:
df_smog.isna().info()

### Opis typu danych w kolumnach

In [ ]:
print(df_smog.dtypes)

### Podgląd danych po zmianie formatowania daty i czasu

In [ ]:
print(df_smog['TYLKO_CZAS'].head())

### Podgląd nazw nagłówków w tabeli

In [ ]:
df_smog.head(2)

### Podstawowe statystyki opisowe

In [ ]:
df_smog.describe()

### Statystyki

In [ ]:
display(df_smog.describe())

### Suma braków

In [ ]:
print(df_smog.isnull().sum())

### Pokaż dane temperatury

In [ ]:
df_smog.sort_values(['TEMPERATURE_AVG']).head(20)

### Histogram wartości danych

In [ ]:
df_smog['TYLKO_CZAS'] = df_smog['TYLKO_CZAS'].astype(str)
skim(df_smog)

### Usuwanie wierszy z brakami w lokalizacjach

In [ ]:
df_smog = df_smog[(df_smog['LATITUDE'] != 0) & (df_smog['LONGITUDE'] != 0)]

### Mapa braków

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_smog.isna(), cbar=False, cmap='viridis')
plt.title('Mapa kompletności zbioru danych (kolor jednolity = brak pustych wartości)', fontsize=14)
plt.show()

Powyższa mapa cieplna (heatmapa) potwierdza całkowitą kompletność analizowanego zbioru danych. Jednolity kolor oraz brak odchyleń na skali udowadniają, że w żadnej z kolumn nie występują już puste wartości.

### Analiza kompletności danych

In [ ]:
msno.matrix(df_smog)

In [ ]:
md(f"Zbiór danych obejmujący {df_smog.shape[0]} wierszy cechuje się bardzo wysoką kompletnością. Macierz braków ujawniła punktowy, jednoczesny brak odczytów dla zmiennych meteorologicznych (wilgotność, ciśnienie, temperatura) w ułamku obserwacji. Ze względu na zachowanie ciągłości pomiarów zanieczyszczeń (PM) w tych samych wierszach, anomalia wskazuje na jednostkowy błąd czujnika pogody.")

#### Uzupełnienie pustych miejsc (NaN) ostatnią prawidłowo odczytaną wartością (forward fill)

In [ ]:
df_smog[['HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG']] = df_smog[['HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG']].ffill()

### Macierz braków danych

In [ ]:
msno.matrix(df_smog)

#### Potwierdzenie braku braków

In [ ]:
msno.heatmap(df_smog)

### Dendrogram

In [ ]:
msno.dendrogram(df_smog)

Wykres hierarchiczny korelacji braków przyjmuje postać płaskiej linii. Potwierdza to jednoznacznie, że po przeprowadzonym procesie czyszczenia danych, w zbiorze nie ostały się żadne powiązane ze sobą luki w odczytach. Zbiór jest w 100% kompletny.

### Check, if every area is in place (longitude and latitude are in Polish area)

In [ ]:
# Define a function and area latitude and longitude min and max values
for y, x in df_smog['LATITUDE'].items() and df_smog['LONGITUDE'].items():
    x = float(x)

def check_areas(df):
    area_values = {
        "Latitude.North < 54.835563": df_smog['LATITUDE'] < 54.8,
        "Latitude.South > 49.002063": df_smog['LATITUDE'] > 49.0,
        "Longitude.East < 24.145562": df_smog['LONGITUDE'] < 24.2,
        "Longitude.West > 14.124562": df_smog['LONGITUDE'] > 14.0
        }
    return area_values

# Define an object table with the area checker

areas = check_areas(df_smog)

# Print if there are any differences in the areas

for rule, result in areas.items():
    print(f"{rule}: {not result.all()}")

### Check, how many of areas have wrong latitudes\longitudes

In [ ]:
# Check, how many differences are in areas

differences = {rule: ~result for rule, result in areas.items()}
summary = {rule: result.sum() for rule, result in differences.items()}

# Print the number of differences

for rule, count in summary.items():
    print(f"{rule}: {count} differences")

### Analiza Najbardziej Zanieczyszczonych Obszarów

In [ ]:
plt.figure(figsize=(12, 6))
top_15_smog = df_smog.groupby('POST_CODE')['PM10_AVG'].mean().nlargest(15).reset_index()
sns.barplot(data=top_15_smog, x='POST_CODE', y='PM10_AVG', color='crimson')
plt.title('Top 15 kodów pocztowych z najwyższym średnim stężeniem PM10', fontsize=14)
plt.ylabel('Średnie PM10 (µg/m³)')
plt.xlabel('Kod Pocztowy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Rozkład zanieczyszczeń w ujęciu wojewódzkim

In [ ]:
plt.figure(figsize=(10, 8))

# Sortowanie województw od najbardziej do najmniej zanieczyszczonych
kolejnosc = df_smog.groupby('AREA')['PM10_AVG'].mean().sort_values(ascending=False).index

sns.barplot(data=df_smog, y='AREA', x='PM10_AVG', errorbar=None, color='steelblue', order=kolejnosc)

plt.title('Średnie stężenie PM10 w podziale na województwa', fontsize=14)
plt.xlabel('Średnie stężenie PM10 (µg/m³)')
plt.ylabel('Województwo')
plt.tight_layout()
plt.show()

### Analiza korelacji: Temperatura a stężenie PM10

In [ ]:
df_czyste_temp = df_smog[df_smog['TEMPERATURE_AVG'] > 0]
plt.figure(figsize=(12, 7))

# 1. Zmniejszamy kropki (dodajemy parametr scatter_kws z mniejszym 's' i mniejszą 'alpha')
sns.regplot(data=df_czyste_temp, x='TEMPERATURE_AVG', y='PM10_AVG', 
            scatter_kws={'alpha':0.1, 's':15, 'color':'steelblue'}, 
            line_kws={'color':'crimson'}, label='PM10')

sns.regplot(data=df_czyste_temp, x='TEMPERATURE_AVG', y='PM25_AVG', 
            scatter_kws={'alpha':0.1, 's':15, 'color':'mediumseagreen'}, 
            line_kws={'color':'darkorange'}, label='PM2.5')

# 2. ODCINAMY OŚ Y! Pokazujemy tylko wartości od 0 do 100 µg/m³
plt.ylim(-5, 100) 

plt.title('Zależność między temperaturą a stężeniem pyłów (z powiększeniem głównego skupiska)', fontsize=14)
plt.xlabel('Średnia Temperatura (°C)')
plt.ylabel('Stężenie Pyłów (µg/m³)')
plt.legend()
plt.tight_layout()
plt.show()

### Analiza korelacji PM10 i PM2.5 z temperaturą:
- Brak silnego związku z temperaturą. Linie trendu (czerwona i pomarańczowa) pną się lekko w górę a szeroka "chmura" punktów wskazuje, że sama temperatura nie jest głównym czynnikiem napędzającym smog w lipcu.
- Wysoka jakość powietrza - większość odczytów (duże zagęszczenie na samym dole wykresu) znajduje się poniżej bezpiecznego maksimum 50 µg/m³.
- Ścisła współzależność frakcji: Kropki zielone (PM2.5) i niebieskie (PM10) oraz ich linie trendu nakładają się na siebie niemal idealnie, co potwierdza, że pochodzą z tych samych źródeł emisji.

### Analiza regionalna zanieczyszczeń (PM10 vs PM2.5)

In [ ]:
plt.figure(figsize=(12, 8))
df_grouped = df_smog.groupby('AREA')[['PM10_AVG', 'PM25_AVG']].mean().sort_values(by='PM10_AVG', ascending=False).reset_index()
df_melted = df_grouped.melt(id_vars='AREA', var_name='Frakcja', value_name='Średnie stężenie')
sns.barplot(data=df_melted, y='AREA', x='Średnie stężenie', hue='Frakcja', palette=['steelblue', 'mediumseagreen'])
plt.title('Porównanie stężenia frakcji PM10 i PM2.5 w podziale na województwa', fontsize=14)
plt.xlabel('Średnie stężenie (µg/m³)')
plt.ylabel('Województwo')
plt.legend(title='Frakcja pyłu')
plt.tight_layout()
plt.show()

In [ ]:
sns.barplot(data=df_smog,x='TEMPERATURE_AVG',y='PM25_AVG',errorbar=None,estimator=np.mean,color='steelblue')

In [ ]:
plt.figure(figsize=(10, 8))
korelacje = df_smog.corr(numeric_only=True)
sns.heatmap(korelacje, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Macierz korelacji zmiennych numerycznych', fontsize=14)
plt.tight_layout()
plt.show()

### Usuwanie duplikatów

In [ ]:
korelacje_lista = korelacje.unstack().sort_values(key=abs, ascending=False).drop_duplicates()
korelacje_lista = korelacje_lista[korelacje_lista != 1.0]
print("Najsilniejsze znalezione korelacje w danych:")

In [ ]:
print(korelacje_lista.head(10))

In [ ]:
md(f"# Analiza korelacji: \n- Potwierdzona zostaje dominująca zależność między frakcjami PM2.5 a PM10 ({korelacje_lista.iloc[0]:.2f}). \n- Na drugim miejscu plasuje się silna ujemna korelacja ({korelacje_lista.iloc[1]:.2f}) między wilgotnością a temperaturą.\nPojawiła się również zależność geograficzna:\n- Długość i szerokość geograficzna ({korelacje_lista.iloc[2]:.2f}) sugerują, że ułożenie stacji ma pewien matematyczny wpływ na układ danych.")

### Histogram dla zapylenia powietrza

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist([df_smog['PM10_AVG'], df_smog['PM25_AVG']], bins=40, color=['steelblue', 'mediumseagreen'], label=['PM10', 'PM2.5'], edgecolor='black')
plt.xlim(0, 100)
plt.title('Porównanie rozkładu stężeń PM10 i PM2.5 w analizowanym okresie', fontsize=14)
plt.ylabel('Liczba pomiarów')
plt.xlabel('Wartość stężenia (µg/m³)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Zbieżność kształtu rozkładu: Obie frakcje charakteryzują się identycznym, prawoskośnym kształtem rozkładu. 
Oba pyły zachowują się w środowisku w ten sam sposób i podlegają tym samym trendom

### Analiza współzależności frakcji pyłów

In [ ]:
plt.figure(figsize=(10, 8))
sns.regplot(data=df_smog, x='PM10_AVG', y='PM25_AVG', 
            scatter_kws={'alpha': 0.1, 'color': 'steelblue'}, 
            line_kws={'color': 'crimson'})
plt.title('Zależność stężenia PM2.5 od PM10', fontsize=14)
plt.xlabel('Wartość PM10 (µg/m³)')
plt.ylabel('Wartość PM2.5 (µg/m³)')
plt.grid(linestyle='--', alpha=0.5)
plt.show()

Analiza potwierdza ekstremalnie silną korelację.
Punkty układają się w bardzo wąskim, zwartym paśmie wzdłuż linii regresji. Oznacza to, że proporcja szkodliwszego PM2.5 w ogólnej puli PM10 jest wysoce przewidywalna i stała.

### Wykres liniowy - zmiana stężenia pyłów zawieszonych (PM10 i PM2.5) na przestrzeni badanego tygodnia (17-23 lipca 2026 r.)

In [ ]:
plt.figure(figsize=(14, 6))
df_smog['TIMESTAMP'] = pd.to_datetime(df_smog['TIMESTAMP'], errors='coerce')
sns.lineplot(data=df_smog, x='TIMESTAMP', y='PM10_AVG', label='PM10', color='steelblue', errorbar=None)
sns.lineplot(data=df_smog, x='TIMESTAMP', y='PM25_AVG', label='PM2.5', color='mediumseagreen', errorbar=None)
plt.title('Przebieg zanieczyszczeń powietrza w czasie (Szereg czasowy)', fontsize=14)
plt.xlabel('Data i godzina')
plt.ylabel('Stężenie Pyłów (µg/m³)')
plt.xticks(rotation=45) 
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Frakcja')
plt.tight_layout()
plt.show()

### Wykres pudełkowy (boxplot) dla regionów

In [ ]:
plt.figure(figsize=(12, 6))
kolejnosc = df_smog.groupby('AREA')['PM25_AVG'].median().sort_values(ascending=False).index
sns.boxplot(x='AREA', y='PM25_AVG', data=df_smog, order=kolejnosc, palette='viridis', hue='AREA', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Rozkład stężenia PM2.5 w województwach (Mediana i wartości odstające)', fontsize=14)
plt.ylabel('Stężenie PM2.5 (µg/m³)')
plt.xlabel('Województwo')
plt.tight_layout()
plt.show()

Pudełka są bardzo płaskie i leżą na samym dole wykresu. Oznacza to, że przez większość czasu (w 50% pomiarów) powietrze w każdym województwie jest bardzo czyste i stabilne.
Czarne kropki (wartości odstające / outliery) Pokazują pojedyncze, ekstremalne strzały smogu.

### Definiowanie funkcji do obliczenia współczynnika zmienności (w procentach)


In [ ]:
def wsp_zmiennosci(x):
    return (x.std() / x.mean()) * 100

### Wywoływanie interesujących nas statystyk dla konkretnej kolumny (lub kilku kolumn)

In [ ]:
def q1(x): return x.quantile(0.25)
def q3(x): return x.quantile(0.75)
statystyki = df_smog[['PM10_AVG', 'PM25_AVG']].agg([
    'mean',           # Średnia
    'median',         # Mediana
    'std',            # Odchylenie standardowe
    wsp_zmiennosci,   # Współczynnik zmienności (nasza funkcja zdefiniowana wyżej)
    q1,               # Pierwszy kwartyl
    q3                # Trzeci kwartyl
])
statystyki.index = ['Średnia', 'Mediana', 'Odch. standardowe', 'Wsp. zmienności (%)', 'Kwartyl 1 (Q1)', 'Kwartyl 3 (Q3)']
display(statystyki.round(2))

### Najwyższe zanieczyszczenie PM10

In [ ]:
print("Top 5 najbardziej zanieczyszczonych pomiarów (PM10):")
kolumny_do_pokazania = ['TIMESTAMP', 'CITY', 'AREA', 'NAME', 'PM10_AVG', 'PM25_AVG']
top_5_smog = df_smog.nlargest(5, 'PM10_AVG')[kolumny_do_pokazania]

display(top_5_smog)

### Rozkład godzinowy stężenia PM10 i PM2.5 - najwyższe stężenie w ciągu doby.

In [ ]:
df_smog['TIMESTAMP'] = pd.to_datetime(df_smog['TIMESTAMP'], errors='coerce')
df_smog['HOUR'] = df_smog['TIMESTAMP'].dt.hour

# 2. Rysujemy wykres używając nowej kolumny 'HOUR'
plt.figure(figsize=(12, 6))

# Grupowanie po godzinie (HOUR), a nie TYLKO_CZAS
hourly_trend = df_smog.groupby('HOUR')[['PM10_AVG', 'PM25_AVG']].mean().reset_index()

sns.lineplot(x='HOUR', y='PM10_AVG', data=hourly_trend, marker='o', label='PM10', color='crimson')
sns.lineplot(x='HOUR', y='PM25_AVG', data=hourly_trend, marker='s', label='PM2.5', color='navy')

plt.title('Średni dobowy profil stężenia pyłów zawieszonych', fontsize=14)
plt.xlabel('Godzina (0-23)')
plt.ylabel('Średnie Stężenie (µg/m³)')

# Teraz to zadziała, bo na osi X mamy liczby całkowite od 0 do 23
plt.xticks(range(0, 24)) 
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### MAPA ZANIECZYSZCZEŃ

In [ ]:
map_ts = dt.datetime.now().strftime("%Y%m%d_%H-%M")
output_map = f"../data/mapa_szkol{map_ts}.html"
df_schools = df_smog.groupby(['NAME', 'LATITUDE', 'LONGITUDE'])['PM10_AVG'].mean().reset_index()
m = folium.Map(location=[52.0, 19.0], zoom_start=6)
for idx, row in df_schools.iterrows():
    # Definiujemy kolory ostrzegawcze
    if row['PM10_AVG'] <= 20:
        color = 'green'
    elif row['PM10_AVG'] <= 50:
        color = 'orange'
    else:
        color = 'red'
        
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=6,
        popup=f"<b>{row['NAME']}</b><br>Średnie PM10: {row['PM10_AVG']:.1f} µg/m³",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
    ).add_to(m)
m.save(output_map)
m

### Wnioski z przeprowadzonych badań

In [ ]:
md(f"# Wyniki dla PM10:\nŚrednia (mean) wynosi ok. {statystyki.loc['Średnia', 'PM10_AVG']:.2f}, podczas gdy mediana (median) to zaledwie {statystyki.loc['Mediana', 'PM10_AVG']:.2f}. \n- Interpretacja: Mediana mówi nam, że w dokładnie połowie badanych przypadków zanieczyszczenie pyłem PM10 wynosiło {statystyki.loc['Mediana', 'PM10_AVG']:.2f} lub mniej. Dobowa norma alarmowa to 50. Ponieważ średnia jest zauważalnie wyższa od mediany, mamy tu do czynienia z rozkładem silnie prawoskośnym. Oznacza to, że przez większość czasu powietrze jest bardzo czyste, ale występują nagłe, rzadkie godziny z gigantycznym smogiem, które matematycznie \"ciągną\" średnią w górę.")

In [ ]:
md(f"Odchylenie standardowe (std) wynosi {statystyki.loc['Odch. standardowe', 'PM10_AVG']:.2f}, a współczynnik zmienności osiągnął {statystyki.loc['Wsp. zmienności (%)', 'PM10_AVG']:.2f}%\n- Interpretacja: Współczynnik zmienności powyżej 100% oznacza ekstremalnie wysoki rozrzut danych. Odchylenie standardowe jest prawie dwukrotnie wyższe od samej średniej. Zjawisko smogu jest niestabilne i epizodyczne. Wysoka zmienność wskazuje na niestabilność warunków atmosferycznych.")

In [ ]:
md(f"Dolny kwartyl (Q1) to {statystyki.loc['Kwartyl 1 (Q1)', 'PM10_AVG']:.2f}, a górny kwartyl (Q3) to {statystyki.loc['Kwartyl 3 (Q3)', 'PM10_AVG']:.2f}. \n- Interpretacja: Te liczby oznaczają, że aż połowa (środkowe 50%) wszystkich zebranych pomiarów jest w przedziale między {statystyki.loc['Kwartyl 1 (Q1)', 'PM10_AVG']:.2f} a {statystyki.loc['Kwartyl 3 (Q3)', 'PM10_AVG']:.2f}, Q3 wynosi {statystyki.loc['Kwartyl 3 (Q3)', 'PM10_AVG']:.2f}, co oznacza, że aż 75% wszystkich pomiarów jest niższych niż {statystyki.loc['Kwartyl 3 (Q3)', 'PM10_AVG']:.2f}. Problemy ze smogiem (i za podwyższoną średnią oraz ogromne odchylenie) odpowiadają najwyższe 25% pomiarów.")

In [ ]:
md(f"# Wyniki dla PM2.5:\nŚrednia (mean) wynosi ok. {statystyki.loc['Średnia', 'PM25_AVG']:.2f}, podczas gdy mediana (median) to zaledwie {statystyki.loc['Mediana', 'PM25_AVG']:.2f}. \n- Interpretacja: Mediana mówi nam, że w dokładnie połowie badanych przypadków zanieczyszczenie pyłem PM2.5 wynosiło zaledwie {statystyki.loc['Mediana', 'PM25_AVG']:.2f} lub mniej. Dobowa norma alarmowa to 15 (wg WHO). Ponieważ średnia jest zauważalnie wyższa od mediany, mamy tu do czynienia z rozkładem silnie prawoskośnym. Oznacza to, że przez większość czasu powietrze jest bardzo czyste, ale występują nagłe, rzadkie godziny z gigantycznym smogiem, które matematycznie \"ciągną\" średnią w górę.")

In [ ]:
md(f"Odchylenie standardowe (std) wynosi {statystyki.loc['Odch. standardowe', 'PM25_AVG']:.2f}, a współczynnik zmienności osiągnął {statystyki.loc['Wsp. zmienności (%)', 'PM25_AVG']:.2f}%\n- Interpretacja: Współczynnik zmienności powyżej 100% oznacza ekstremalnie wysoki rozrzut danych. Odchylenie standardowe jest prawie dwukrotnie wyższe od samej średniej. Zjawisko smogu jest niestabilne i epizodyczne. Wysoka zmienność wskazuje na niestabilność warunków atmosferycznych.")

In [ ]:
md(f"Dolny kwartyl (Q1) to {statystyki.loc['Kwartyl 1 (Q1)', 'PM25_AVG']:.2f}, a górny kwartyl (Q3) to {statystyki.loc['Kwartyl 3 (Q3)', 'PM25_AVG']:.2f}. \n- Interpretacja: Te liczby oznaczają, że aż połowa (środkowe 50%) wszystkich zebranych pomiarów jest w przedziale między {statystyki.loc['Kwartyl 1 (Q1)', 'PM25_AVG']:.2f} a {statystyki.loc['Kwartyl 3 (Q3)', 'PM25_AVG']:.2f}, Q3 wynosi {statystyki.loc['Kwartyl 3 (Q3)', 'PM25_AVG']:.2f}, co oznacza, że aż 75% wszystkich pomiarów jest niższych niż {statystyki.loc['Kwartyl 3 (Q3)', 'PM25_AVG']:.2f}. Problemy ze smogiem (i za podwyższoną średnią oraz ogromne odchylenie) odpowiadają najwyższe 25% pomiarów.")

### Save dataframe to csv file **Always put as a last cell !**

In [ ]:
csv_ts = dt.datetime.now().strftime("%Y%m%d_%H-%M")
df_smog.to_csv(f"../tests/smog_raport_raw{csv_ts}.csv", index=False)